In [2]:
# 모듈 불러오기
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# csv파일 읽어오기

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

train.head()

# Id 분리

test_ids = test["Id"]

# 이상치 정리

train = train.drop(train[(train["GrLivArea"]>4000)& (train["SalePrice"]<300000)].index)

print(train.shape)

# 로그 변환

train["SalePrice"] = np.log1p(train["SalePrice"])

#feature Engineering

# TotalSF
train["TotalSF"] = train["TotalBsmtSF"] + train["1stFlrSF"] + train["2ndFlrSF"]
test["TotalSF"] = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]

# TotalBathrooms
train["TotalBathrooms"] = (
    train["FullBath"]
    + train["HalfBath"] * 0.5
    + train["BsmtFullBath"]
    + train["BsmtHalfBath"] * 0.5
)

test["TotalBathrooms"] = (
    test["FullBath"]
    + test["HalfBath"] * 0.5
    + test["BsmtFullBath"]
    + test["BsmtHalfBath"] * 0.5
)

# HouseAge
train["HouseAge"] = train["YrSold"] - train["YearBuilt"]
test["HouseAge"] = test["YrSold"] - test["YearBuilt"]

# TotalRooms
train["TotalRooms"] = train["TotRmsAbvGrd"] + train["BedroomAbvGr"]
test["TotalRooms"] = test["TotRmsAbvGrd"] + test["BedroomAbvGr"]

# 결측치 처리

#LotFrontage
train["LotFrontage"] = train.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

test["LotFrontage"] = test.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

#None cols
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]

for col in none_cols:
    if col in train.columns:
        train[col] = train[col].fillna("None")
    if col in test.columns:
        test[col] = test[col].fillna("None")

#Zerocols
zero_cols = [
    "MasVnrArea", "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath"
]

for col in zero_cols:
    if col in train.columns:
        train[col] = train[col].fillna(0)
    if col in test.columns:
        test[col] = test[col].fillna(0)

#Mode & Median
# train
for col in train.select_dtypes(include=["object", "string"]).columns:
    train[col] = train[col].fillna(train[col].mode()[0])

for col in train.select_dtypes(include=["int64", "float64"]).columns:
    train[col] = train[col].fillna(train[col].median())

# test
for col in test.select_dtypes(include=["object", "string"]).columns:
    test[col] = test[col].fillna(test[col].mode()[0])

for col in test.select_dtypes(include=["int64", "float64"]).columns:
    test[col] = test[col].fillna(test[col].median())

print("train NaN 총개수:", train.isnull().sum().sum())
print("test NaN 총개수:", test.isnull().sum().sum())

#Id 제거

train = train.drop(["Id"],axis=1)
test = test.drop(["Id"],axis=1)

#One-Hot Encoding

train = pd.get_dummies(train)
test = pd.get_dummies(test)

#X,y 분리

y = train["SalePrice"]
X = train.drop("SalePrice",axis=1)

# train/test 컬럼 맞추기

X, test = X.align(test, join="left", axis= 1, fill_value=0)

print("X shape:", X.shape)
print("test shape:", test.shape)
print("SalePrice in X?", "SalePrice" in X.columns)
print("SalePrice in test?", "SalePrice" in test.columns)

#skew feature 찾기 + log 변환

skew = X.select_dtypes(include=["int64","float64"]).skew()
skew = skew[abs(skew)> 0.75]

print(skew.sort_values(ascending=False).head(10))

for col in skew.index:
    X[col] = np.log1p(X[col])
    test[col] = np.log1p(test[col])

(1460, 81)
(1459, 80)
(1458, 81)
train NaN 총개수: 0
test NaN 총개수: 0
X shape: (1458, 305)
test shape: (1459, 305)
SalePrice in X? False
SalePrice in test? False
MiscVal          24.460085
PoolArea         15.948945
LotArea          12.573925
3SsnPorch        10.297106
LowQualFinSF      9.004955
KitchenAbvGr      4.484883
BsmtFinSF2        4.251925
ScreenPorch       4.118929
BsmtHalfBath      4.100114
EnclosedPorch     3.087164
dtype: float64


In [3]:
# RandomForest CV

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

rf_rmse = -rf_scores
print("RandomForest RMSE:", rf_rmse)
print("평균:", rf_rmse.mean())

RandomForest RMSE: [0.13477814 0.13764379 0.14239898 0.13254199 0.13761663]
평균: 0.13699590511735982


In [4]:
#Ridge CV

ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10)
)

ridge_scores = cross_val_score(
    ridge,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

ridge_rmse = -ridge_scores
print("Ridge RMSE:", ridge_rmse)
print("평균:", ridge_rmse.mean())

Ridge RMSE: [0.11721761 0.12763096 0.13735066 0.11092851 0.11988328]
평균: 0.12260220377078786


In [ ]:
#Lasso CV

lasso = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.001, max_iter=10000)
)

lasso_scores = cross_val_score(
    lasso,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

lasso_rmse = -lasso_scores
print("Lasso RMSE:", lasso_rmse)
print("평균:", lasso_rmse.mean())

Lasso RMSE: [0.10762713 0.11815543 0.12669687 0.10823924 0.11410348]
평균: 0.11496442936147384


In [6]:
# LightGBM CV

lgbm = make_pipeline(
    StandardScaler(),
    LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
    )
)

lgbm_scores = cross_val_score(
    lgbm,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

lgbm_rmse = -lgbm_scores
print("LGBM RMSE:", lgbm_rmse)
print("평균:", lgbm_rmse.mean())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001794 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3766
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 199
[LightGBM] [Info] Start training from score 12.021352


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001464 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3772
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 195
[LightGBM] [Info] Start training from score 12.023516


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3774
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 199
[LightGBM] [Info] Start training from score 12.020683


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3789
[LightGBM] [Info] Number of data points in the train set: 1167, number of used features: 196
[LightGBM] [Info] Start training from score 12.032713


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3776
[LightGBM] [Info] Number of data points in the train set: 1167, number of used features: 198
[LightGBM] [Info] Start training from score 12.021807
LGBM RMSE: [0.12118919 0.13164444 0.13295182 0.11152547 0.12283613]
평균: 0.12402940969822848


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
